# 小雅 Kaggle T4 · 单图音频数字人口播 MP4 实际验收

独立试验，与当前 DUIX 无关。本 Notebook 使用 **SadTalker** 将 GitHub 小雅肖像与已在 Kaggle 生成的真实 8.24s 语音结合，尝试产出带音轨的 MP4。最终生产版 DUIX 仍需单独验证。

需要 Settings 开启 GPU T4 与 Internet。GPU 额度由 Kaggle 决定；Notebook 失败不得伪报成功。SadTalker 代码来自 https://github.com/OpenTalker/SadTalker，首次会下载模型权重并可能遇到依赖兼容问题。

In [ ]:
import sys, os, pathlib, time, subprocess, json, hashlib, shutil, traceback
work=pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").exists() else pathlib.Path.cwd()
out=work/"xiaoya_video_results";out.mkdir(parents=True,exist_ok=True)
t0=time.monotonic()
probe=subprocess.run(["nvidia-smi","--query-gpu=name,memory.used,memory.total","--format=csv,noheader,nounits"],capture_output=True,text=True)
print(probe.stdout or probe.stderr)
assert probe.returncode==0,"GPU not enabled"
print("Python",sys.version)


In [ ]:
import subprocess, sys, importlib
# Kaggle uses Python 3.13 / NumPy 2. Do not pin unsupported NumPy<2 or SciPy<1.13.
# Install small, compatible packages in stages; fail immediately on pip errors.
groups=[
  ["facexlib==0.3.0","face-alignment","kornia","yacs","safetensors"],
  ["resampy","pydub","imageio","imageio-ffmpeg","av","librosa","PyYAML","soundfile"],
]
for group in groups:
    cmd=[sys.executable,"-m","pip","install","--disable-pip-version-check","-q",*group]
    result=subprocess.run(cmd,text=True,capture_output=True,timeout=360)
    print("Install",group,":",result.returncode)
    if result.returncode:
        print((result.stdout+"\n"+result.stderr)[-7000:])
        raise RuntimeError("Dependency install failed: "+", ".join(group))
for name in ("facexlib","kornia","yacs","safetensors","librosa","soundfile"):
    importlib.import_module(name)
print("All required lightweight imports verified")


In [ ]:
import requests
from PIL import Image
base="https://raw.githubusercontent.com/mjw3177109/xunigirl/main/"
def download(url,path,minimum=1024):
    path=pathlib.Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists() and path.stat().st_size >= minimum:return path
    r=requests.get(url,timeout=(25,180),stream=True);r.raise_for_status()
    with open(path,"wb") as f:
        for buf in r.iter_content(1024*1024):
            if buf:f.write(buf)
    assert path.stat().st_size>=minimum, f"Download too small: {url}"
    return path
image_file=download(base+"assets/portraits/xiaoya.png",work/"xiaoya.png",150000)
audio_file=download(base+"voices/xiaoya/ref_audio.wav",work/"xiaoya.wav",200000)
audio_sha=hashlib.sha256(audio_file.read_bytes()).hexdigest()
assert audio_sha=="5017add8a805d69b97a60d4a7caf16d0f087c8dbec0cf62e636154b1ee289feb", "Not the validated Kaggle-generated voice"
photo=Image.open(image_file).convert("RGB")
w,h=photo.size
# Fixed-asset, face-centred head-and-shoulders crop: the previous 67% full-body
# crop placed Xiaoya's face near the upper edge and could yield an out-of-bounds
# face-detector box downstream. Only a working copy is cropped.
side=max(256,int(min(w,h)*0.40))
cx=w//2
top=max(0,int(h*0.01))
left=max(0,min(w-side,cx-side//2))
top=max(0,min(h-side,top))
assert 0 <= left < left+side <= w and 0 <= top < top+side <= h
crop=photo.crop((left,top,left+side,top+side)).resize((640,640),Image.Resampling.LANCZOS)
portrait=work/"xiaoya_closeup.png";crop.save(portrait)
assert portrait.is_file() and portrait.stat().st_size > 10000
(out/"input_preflight.json").write_text(json.dumps({
 "original_size":[w,h],"crop_box":[left,top,left+side,top+side],
 "saved_size":list(crop.size),"portrait":str(portrait),
 "voice_sha256":audio_sha
},ensure_ascii=False,indent=2),encoding="utf-8")
print("input",photo.size,"closeup",crop.size,"audio_sha256",audio_sha)


In [ ]:
sadtalker=work/"SadTalker"
if not (sadtalker/"inference.py").exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/OpenTalker/SadTalker.git",str(sadtalker)],check=True,timeout=180)
# Old SadTalker + new torchvision removed this path; patch only a known compatibility import.
import site
for top in [*site.getsitepackages(),site.getusersitepackages()]:
    p=pathlib.Path(top)/"basicsr/data/degradations.py"
    if p.is_file():
        src=p.read_text()
        before="from torchvision.transforms.functional_tensor import rgb_to_grayscale"
        after="from torchvision.transforms.functional import rgb_to_grayscale"
        if before in src:p.write_text(src.replace(before,after))
# NumPy 2 removed legacy aliases used by SadTalker. Patch the downloaded
# open-source project only; do not downgrade Kaggle's Python 3.13 runtime.
import re
patch_counts={}
for pyfile in (sadtalker/"src").rglob("*.py"):
    original=pyfile.read_text(encoding="utf-8")
    fixed=original.replace("np.VisibleDeprecationWarning",
        "np.exceptions.VisibleDeprecationWarning")
    for old,new in [("float","float64"),("int","int64"),("complex","complex128"),("bool","bool_"),("object","object_")]:
        fixed=re.sub(r"\bnp\."+old+r"\b","np."+new,fixed)
    fixed=fixed.replace("Image.ANTIALIAS","Image.Resampling.LANCZOS")
    if fixed!=original:
        pyfile.write_text(fixed,encoding="utf-8")
        patch_counts[str(pyfile.relative_to(sadtalker))]=1
assert patch_counts, "Legacy SadTalker source compatibility patch had no targets"
print("SadTalker compatibility patches:",sorted(patch_counts))
# Upstream SadTalker croper.py slices detections with raw coordinates. A
# negative box edge is a Python negative slice index, producing a *zero-sized*
# input for FAN.get_landmarks -> OpenCV resize !ssize.empty(). Clamp to pixels.
croper=sadtalker/"src/utils/croper.py"
cs=croper.read_text(encoding="utf-8")
old_box="        img = img_np[int(det[1]):int(det[3]), int(det[0]):int(det[2]), :]\n"
new_box=(
 "        height, width = img_np.shape[:2]\n"
 "        x1 = max(0, min(width, int(np.floor(det[0]))))\n"
 "        y1 = max(0, min(height, int(np.floor(det[1]))))\n"
 "        x2 = max(0, min(width, int(np.ceil(det[2]))))\n"
 "        y2 = max(0, min(height, int(np.ceil(det[3]))))\n"
 "        if x2 <= x1 or y2 <= y1:\n"
 "            raise ValueError(f'Invalid face bounding box {det[:4]} for image {width}x{height}')\n"
 "        img = img_np[y1:y2, x1:x2, :]\n"
 "        if img.size == 0:\n"
 "            raise ValueError(f'Empty image crop from face bounding box {det[:4]}')\n"
)
assert cs.count(old_box)==1, "Unexpected SadTalker face-cropping implementation"
cs=cs.replace(old_box,new_box,1)
for old,new in [
 ("lm[:,0] += int(det[0])","lm[:,0] += x1"),
 ("lm[:,1] += int(det[1])","lm[:,1] += y1"),
]:
    assert cs.count(old)==1, "Unexpected landmark-offset code: "+old
    cs=cs.replace(old,new,1)
# Retry the single-face detection at a moderate confidence if strict detection
# fails; never manufacture or fake landmarks.
old_detect="            dets = self.predictor.det_net.detect_faces(img_np, 0.97)"
assert cs.count(old_detect)==1, "Face detection call changed upstream"
cs=cs.replace(old_detect,old_detect+"\n            if len(dets) == 0:\n                dets = self.predictor.det_net.detect_faces(img_np, 0.85)",1)
croper.write_text(cs,encoding="utf-8")
print("Validated clipped face boxes, corrected landmark offsets, and fallback detection")

# Report missing faces as a useful diagnostic, not a Python 3 invalid exception.
preprocess=sadtalker/"src/utils/croper.py"
# Image input is validated independently by the earlier asset preflight.

# Basic MP4 proof has no face enhancement. Legacy GFPGAN and BasicSR
# dependencies are optional; the upstream module eagerly imports them.
enhancer=sadtalker/"src/utils/face_enhancer.py"
es=enhancer.read_text(encoding="utf-8")
eager="from gfpgan import GFPGANer"
guard="try:\n    from gfpgan import GFPGANer\nexcept (ImportError, ModuleNotFoundError):\n    GFPGANer = None"
if eager in es:
    enhancer.write_text(es.replace(eager,guard,1),encoding="utf-8")
print("Face enhancer disabled for low-resolution pipeline verification")
models=sadtalker/"checkpoints";models.mkdir(exist_ok=True)
release="https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/"
weights={
 "SadTalker_V0.0.2_256.safetensors":250000000,
 "mapping_00229-model.pth.tar":10000000,
}
for name,minimum in weights.items():
    path=download(release+name,models/name,minimum)
    print(name,path.stat().st_size)


In [ ]:
import torch
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"]="1"
start=time.monotonic()
args=[sys.executable,"inference.py",
 "--driven_audio",str(audio_file),
 "--source_image",str(portrait),
 "--result_dir",str(out/"render"),
 "--size","256","--batch_size","1",
 "--preprocess","crop","--still"]
env=os.environ.copy()
env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"]="1"
res=subprocess.run(args,cwd=str(sadtalker),env=env,text=True,capture_output=True,timeout=2400)
(out/"inference_stdout.txt").write_text(res.stdout or "",encoding="utf-8")
(out/"inference_stderr.txt").write_text(res.stderr or "",encoding="utf-8")
print("SadTalker exit",res.returncode,"elapsed_seconds",round(time.monotonic()-start,2))
print("stdout last:",(res.stdout or "")[-2000:])
print("stderr last:",(res.stderr or "")[-2600:])
if res.returncode!=0:raise RuntimeError("SadTalker inference failed. Inspect saved stderr/stdout in Kaggle Output.")
videos=sorted((out/"render").rglob("*.mp4"),key=lambda p:p.stat().st_size,reverse=True)
assert videos,"No MP4 output despite zero process exit"
shutil.copyfile(videos[0],out/"xiaoya_talking_pilot.mp4")
check=subprocess.run(["ffprobe","-v","error","-show_entries","format=duration:stream=codec_type,codec_name","-of","json",str(out/"xiaoya_talking_pilot.mp4")],capture_output=True,text=True,check=True)
meta=json.loads(check.stdout)
streams=[s["codec_type"] for s in meta.get("streams",[])]
duration=float(meta["format"]["duration"])
assert "video" in streams and "audio" in streams and 5<=duration<=15, f"Invalid MP4 streams/duration: {meta}"
report={"status":"actual_mp4_generated","engine":"SadTalker (NOT DUIX)","gpu":probe.stdout.strip(),
 "audio_sha256":audio_sha,"seconds":round(duration,3),"streams":meta.get("streams"),
 "elapsed_seconds":round(time.monotonic()-t0,2),"mp4_size_bytes":(out/"xiaoya_talking_pilot.mp4").stat().st_size}
(out/"video_report.json").write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
print(report)
